# Dim tipo unidade 

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window 

In [0]:
df_tru = spark.read.table('saude_sus.trusted.tru_estabelecimento')

In [0]:
display(df_tru)

In [0]:
window_spec = Window.partitionBy("COD_TIPO_UNIDADE").orderBy(sf.col("DAT_ATUALIZACAO").desc())


In [0]:
df_dim = ( 
    df_tru
    .where(sf.col("COD_TIPO_UNIDADE").isNotNull())
    .withColumn("RN", sf.row_number().over(window_spec))
    .where(sf.col("RN") == 1)
    .select(
        sf.md5(
            sf.concat_ws(
                "-",
                sf.col("COD_TIPO_UNIDADE"),
                sf.col("DSC_TIPO_UNIDADE")
            )
        ).alias("SK_TIPO_UNIDADE"),
        sf.col("COD_TIPO_UNIDADE").alias("COD_TIPO_UNIDADE"),
        sf.coalesce(sf.col("DSC_TIPO_UNIDADE"), sf.lit("DESCONHECIDO")).alias("DSC_TIPO_UNIDADE")
    )
)

In [0]:
df_dim.write.mode("overwrite").saveAsTable("saude_sus.refined.dim_tipo_unidade")